# Customer Churn Prediction

This notebook builds a model to predict customer churn (`Exited`) for a bank, using the
`Customer-Churn-Records.csv` dataset.

**Workflow**
1. Load and inspect the data
2. Check for data leakage and drop unusable columns
3. Exploratory data analysis (univariate & bivariate)
4. Encode categorical features
5. Detect outliers and engineer a new feature
6. Fix skewed features
7. Train/test split and baseline models
8. Tune XGBoost and Random Forest
9. Compare models (accuracy, F1, ROC-AUC)
10. Optimize the classification threshold
11. Save the final model for deployment

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('../data/Customer-Churn-Records.csv')

A quick look at the raw data and its structure.

In [ ]:
df.sample(5)

In [ ]:
df.info()

In [ ]:
df.dtypes

**Duplicate rows:**

In [ ]:
df.duplicated().sum()

**Target class balance** — churn (`Exited`) is imbalanced, which we'll need to account for during modeling.

In [ ]:
df['Exited'].value_counts(normalize=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))

sns.countplot(x='Exited', data=df)

plt.title("Customer Churn Distribution")
plt.xlabel("Exited")
plt.ylabel("Number of Customers")

plt.savefig("../images/churn_distribution.png",
            dpi=300,
            bbox_inches="tight")

plt.show()

## 2. Data Leakage Check & Cleaning

In [ ]:
df.select_dtypes(include=['number']).corr()

In [ ]:
print(df[['Complain', 'Exited']].corr())

**Observation — data leakage:** `Complain` is almost perfectly correlated with `Exited`, meaning it
effectively reveals the target and would leak information into the model. `RowNumber`, `CustomerId`,
and `Surname` are identifiers with no predictive value. All four columns are dropped.

In [ ]:
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Complain'])

In [ ]:
df

## 3. Exploratory Data Analysis

### 3.1 Univariate Analysis

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['Age'])

plt.title('Distribution of Age')
plt.xlabel('Age')
plt.ylabel('Count')

**Observation:** the age distribution is right-skewed.

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['CreditScore'])

plt.title('Distribution of Credit Score')
plt.xlabel('Credit Score')
plt.ylabel('Count')

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['Tenure'])

plt.title('Distribution of Tenure')
plt.xlabel('Tenure')
plt.ylabel('Count')

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['Balance'])

plt.title('Distribution of Balance')
plt.xlabel('Balance')
plt.ylabel('Count')

**Observation:** the balance distribution is bimodal (a large spike at zero, plus a roughly normal group of active balances).

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['EstimatedSalary'])

plt.title('Distribution of Estimated Salary')
plt.xlabel('Estimated Salary')
plt.ylabel('Count')

Boxplots of the numerical features, to spot potential outliers at a glance.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15,8))

sns.boxplot(y=df['Age'], ax=axes[0,0])
axes[0,0].set_title('Boxplot of Age')

sns.boxplot(y=df['CreditScore'], ax=axes[0,1])
axes[0,1].set_title('Boxplot of Credit Score')

sns.boxplot(y=df['Tenure'], ax=axes[0,2])
axes[0,2].set_title('Boxplot of Tenure')

sns.boxplot(y=df['Balance'], ax=axes[1,0])
axes[1,0].set_title('Boxplot of Balance')

sns.boxplot(y=df['EstimatedSalary'], ax=axes[1,1])
axes[1,1].set_title('Boxplot of EstimatedSalary')

axes[1,2].set_visible(False)
plt.tight_layout()
plt.show()

Counts of the key categorical features.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

sns.countplot(data=df, x='Geography', ax=axes[0,0])
axes[0,0].set_title('Count of Geography')

sns.countplot(data=df, x='Gender', ax=axes[0,1])
axes[0,1].set_title('Count of Gender')

sns.countplot(data=df, x='Card Type', ax=axes[0,2])
axes[0,2].set_title('Count of Card Type')

sns.countplot(data=df, x='HasCrCard', ax=axes[1,0])
axes[1,0].set_title('Count of HasCrCard')

sns.countplot(data=df, x='IsActiveMember', ax=axes[1,1])
axes[1,1].set_title('Count of IsActiveMember')

sns.countplot(data=df, x='NumOfProducts', ax=axes[1,2])
axes[1,2].set_title('Count of Number of products')

plt.tight_layout()
plt.show()

**Observation:** customers from Germany exhibit the highest churn rate (32.4%), which is nearly
double that of France (16.2%) and Spain (16.7%). This suggests `Geography` is associated with customer
churn and may be an important predictor for the model.

In [ ]:
sns.countplot(data=df, x='Exited')
plt.xticks([0, 1], ['0 (Stayed)', '1 (Churned)'])
plt.title('Target Class Distribution')
plt.show()

print(df['Exited'].value_counts(normalize=True) * 100)

### 3.2 Bivariate Analysis

Comparing the distribution of numerical features between customers who churned and those who
stayed. Boxplots visualize the differences, and mean/median tables quantify them.

In [ ]:
numerical_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary', 'Point Earned']

In [ ]:
for feature in numerical_features:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, x='Exited', y=feature)
    plt.title(f'{feature} vs Exited')
    plt.xlabel('Exited')
    plt.ylabel(feature)
    plt.show()

In [ ]:
### Mean values for each numerical feature
for feature in numerical_features:
    print(f"\n{feature}")
    print(df.groupby('Exited')[feature].mean())

In [ ]:
### Median values for each numerical feature
for feature in numerical_features:
    print(f"\n{feature}")
    print(df.groupby('Exited')[feature].median())

Now the same comparison for the key categorical features.

In [ ]:
categorical_features = ['Geography', 'Gender', 'HasCrCard', 'IsActiveMember']

In [ ]:
for feature in categorical_features:
    plt.figure(figsize=(6,4))
    sns.countplot(data=df, x='Exited', hue=feature)
    plt.title(f'{feature} vs Exited')
    plt.xlabel('Exited')
    plt.ylabel(feature)
    plt.show()

In [ ]:
plt.figure(figsize=(10,8))
corr = df[numerical_features + ['Exited']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

## 4. Feature Encoding

In [ ]:
print(df['Card Type'].unique())

`Card Type` has a natural order (Silver < Gold < Platinum < Diamond), so it's ordinal-encoded rather than one-hot encoded.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

card_order = ['SILVER', 'GOLD', 'PLATINUM', 'DIAMOND']

oe = OrdinalEncoder(categories=[card_order])
df['Card Type'] = oe.fit_transform(df[['Card Type']])

In [ ]:
df

`Geography` has no inherent order, so it's one-hot encoded (dropping the first category to avoid multicollinearity).

In [ ]:
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

In [ ]:
df

`Gender` is binary, so it's mapped directly to 0/1.

In [ ]:
df['Gender'] = df['Gender'].map({'Male':1, 'Female':0})

In [ ]:
df

## 5. Outlier Detection

Checking outliers (via the IQR method) in the features flagged by the earlier boxplots.

In [ ]:
def outlier_detec(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    iqr = Q3-Q1
    lower = Q1 - 1.5*iqr
    upper = Q3 + 1.5*iqr
    outlier = df[(df[col]<lower) | (df[col]>upper)]
    print(f" {col} ")
    print(f"Lower bound: {lower:.2f}, Upper bound: {upper:.2f}")
    print(f"Number of outliers: {len(outlier)} ({len(outlier)/len(df)*100:.2f}%)")
    return outlier

for col in ['CreditScore', 'Age', 'Balance']:
    outliers = outlier_detec(df, col)
    print()

## 6. Feature Engineering

Adding a flag for customers with a zero balance, since the earlier bimodal distribution suggests this group behaves differently.

In [ ]:
df['HasZeroBalance'] = (df['Balance'] == 0).astype(int)

In [ ]:
df

In [ ]:
print(df['HasZeroBalance'].value_counts())
print(df['HasZeroBalance'].value_counts(normalize=True) * 100)

In [ ]:
print(df.groupby('HasZeroBalance')['Exited'].mean())

### Fixing the Age skew

Checking normality of `Age` with a Q-Q plot before deciding whether to transform it.

In [ ]:
import scipy.stats as stats

plt.figure(figsize=(6,4))
stats.probplot(df['Age'], dist="norm", plot=plt)
plt.title('Q-Q Plot: Age')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,5))

stats.probplot(df['Age'], dist="norm", plot=axes[0])
axes[0].set_title('Before Log Transf')

stats.probplot(np.log1p(df['Age']), dist="norm", plot=axes[1])
axes[1].set_title('After Log Transf')

plt.tight_layout()
plt.show()

The log transform noticeably improves normality, so it's applied to `Age`.

In [ ]:
df['Age'] = np.log1p(df['Age'])

In [ ]:
df

## 7. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Exited'])
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 8. Baseline Modeling

### 8.1 Logistic Regression & Decision Tree (no scaling)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

clf = LogisticRegression(max_iter=1000)
clf2 = DecisionTreeClassifier(random_state=42)

clf.fit(X_train, y_train)
clf2.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_pred1 = clf2.predict(X_test)

print("BASELINE (no scaling)")
print("Accuracy LR:", accuracy_score(y_test, y_pred))
print("Accuracy DT:", accuracy_score(y_test, y_pred1))

### 8.2 Linear Regression as a sanity-check classifier

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import f1_score

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred_lin = (lin_reg.predict(X_test) >= 0.5).astype(int)

print("Accuracy Linear Reg (thresholded):", accuracy_score(y_test, y_pred_lin))
print("F1 Linear Reg:", f1_score(y_test, y_pred_lin))

### 8.3 Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

continuous_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary', 'Point Earned']

scaler = StandardScaler()

X_train_final = X_train.copy()
X_test_final = X_test.copy()

X_train_final[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_final[continuous_cols] = scaler.transform(X_test[continuous_cols])

### 8.4 Logistic Regression & Decision Tree (with scaling)

In [ ]:
clf_t = LogisticRegression(max_iter=1000)
clf2_t = DecisionTreeClassifier(random_state=42)

clf_t.fit(X_train_final, y_train)
clf2_t.fit(X_train_final, y_train)

y_pred_t = clf_t.predict(X_test_final)
y_pred1_t = clf2_t.predict(X_test_final)

print("AFTER: scaling")
print("Accuracy LR:", accuracy_score(y_test, y_pred_t))
print("Accuracy DT:", accuracy_score(y_test, y_pred1_t))

### 8.5 Class Imbalance

In [ ]:
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

In [ ]:
from sklearn.metrics import classification_report

clf_bal = LogisticRegression(max_iter=1000, class_weight='balanced')
clf_bal.fit(X_train_final, y_train)
y_pred_bal = clf_bal.predict(X_test_final)

print("LOGISTIC REGRESSION - class_weight='balanced'")
print(classification_report(y_test, y_pred_bal))

## 9. XGBoost

In [ ]:
%pip install xgboost

In [ ]:
import sys
print(sys.executable)

### 9.1 Baseline XGBoost

In [ ]:
from xgboost import XGBClassifier

neg, pos = y_train.value_counts()[0], y_train.value_counts()[1]
spw = neg / pos
print("scale_pos_weight:", spw)

clf_xgb = XGBClassifier(
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=42
)
clf_xgb.fit(X_train_final, y_train)
y_pred_xgb = clf_xgb.predict(X_test_final)
from sklearn.metrics import accuracy_score

xgb_acc = accuracy_score(y_test, y_pred_xgb)
print("Accuracy of XGBoost:", xgb_acc)

print("XGBOOST - scale_pos_weight applied")
print(classification_report(y_test, y_pred_xgb))

### 9.2 Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=spw
)

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 0.1, 0.2]
}

xgb_random = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

xgb_random.fit(X_train_final, y_train)

best_xgb = xgb_random.best_estimator_

print("Best XGBoost Parameters:")
print(xgb_random.best_params_)

### 9.3 Feature Importance

In [ ]:
import pandas as pd

feature_importance = pd.Series(
    best_xgb.feature_importances_,
    index=X_train_final.columns
).sort_values(ascending=False)

print("Feature Importance:")
print(feature_importance)

In [ ]:
plt.figure(figsize=(10, 6))

feature_importance.head(10).sort_values().plot(kind="barh")

plt.title("Top 10 Features Influencing Customer Churn")
plt.xlabel("Feature Importance")
plt.ylabel("Feature")

plt.tight_layout()

plt.savefig("../images/xgboost_feature_importance.png",
            dpi=300,
            bbox_inches="tight")

plt.show()

## 10. Random Forest

### 10.1 Baseline Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall   :", recall_score(y_test, y_pred_rf))
print("F1 Score :", f1_score(y_test, y_pred_rf))

print("\nClassification Report")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_rf))

### 10.2 Hyperparameter Tuning

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestClassifier(random_state=42)

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False]
}

rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

rf_random.fit(X_train_final, y_train)

best_rf = rf_random.best_estimator_

print("Best RF Parameters:")
print(rf_random.best_params_)

## 11. Model Comparison

In [ ]:
y_pred_rf = best_rf.predict(X_test_final)
y_pred_xgb = best_xgb.predict(X_test_final)

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred1),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ]
})

results["Accuracy (%)"] = (results["Accuracy"] * 100).round(2)
results = results.sort_values(by="Accuracy", ascending=False)

print(results)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],

    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred1),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],

    "F1 Score": [
        f1_score(y_test, y_pred, average="weighted"),
        f1_score(y_test, y_pred1, average="weighted"),
        f1_score(y_test, y_pred_rf, average="weighted"),
        f1_score(y_test, y_pred_xgb, average="weighted")
    ]
})

print(comparison)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7,5))
plt.bar(comparison["Model"], comparison["Accuracy"])
plt.title("Comparison of Machine Learning Models Based on Accuracy")
plt.xlabel("Models")
plt.ylabel("Accuracy")

for i, value in enumerate(comparison["Accuracy"]):
  plt.text(i, value + 0.01, f"{value:.3f}", ha="center")
plt.ylim(0, 1)
plt.savefig("../images/model_accuracy_comparison.png",
            dpi=300,
            bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,5))
plt.bar(comparison["Model"], comparison["F1 Score"])
plt.title("Comparison of Machine Learning Models Based on Weighted F1 Score")
plt.xlabel("Machine Learning Models")
plt.ylabel("F1 Score")

for i, value in enumerate(comparison["F1 Score"]):
    plt.text(i, value + 0.02, f"{value:.3f}", ha="center", fontsize=11)

plt.ylim(0, 1)
plt.savefig("../images/model_f1score_comparison.png",
            dpi=300,
            bbox_inches="tight")
plt.show()

XGBoost performs best, so its confusion matrix is examined more closely.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred_xgb)
plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Stayed (0)", "Exited (1)"],
    yticklabels=["Stayed (0)", "Exited (1)"]
)

plt.title("Confusion Matrix - XGBoost")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.savefig("../images/confusion_matrix_xgboost.png",
            dpi=300,
            bbox_inches="tight")
plt.show()

## 12. ROC Curve Analysis

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

In [ ]:
# Logistic Regression ROC Curve
y_prob_lr = clf.predict_proba(X_test)[:, 1]
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
auc_lr = auc(fpr_lr, tpr_lr)
print("Logistic Regression AUC:", auc_lr)

In [ ]:
# Decision Tree ROC Curve
y_prob_dt = clf2.predict_proba(X_test)[:, 1]
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
auc_dt = auc(fpr_dt, tpr_dt)
print("Decision Tree AUC:", auc_dt)

In [ ]:
# Random Forest ROC Curve
y_prob_rf = best_rf.predict_proba(X_test_final)[:, 1]
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
auc_rf = auc(fpr_rf, tpr_rf)
print("Random Forest AUC:", auc_rf)

In [ ]:
# XGBoost ROC Curve
y_prob_xgb = best_xgb.predict_proba(X_test_final)[:, 1]
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
auc_xgb = auc(fpr_xgb, tpr_xgb)
print("XGBoost AUC:", auc_xgb)

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(fpr_lr, tpr_lr,
         linewidth=2,
         label=f"Logistic Regression (AUC = {auc_lr:.3f})")

plt.plot(fpr_dt, tpr_dt,
         linewidth=2,
         label=f"Decision Tree (AUC = {auc_dt:.3f})")

plt.plot(fpr_rf, tpr_rf,
         linewidth=2,
         label=f"Random Forest (AUC = {auc_rf:.3f})")

plt.plot(fpr_xgb, tpr_xgb,
         linewidth=2,
         label=f"XGBoost (AUC = {auc_xgb:.3f})")

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5,
         label="Random Classifier")

plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curve Comparison of Machine Learning Models", fontsize=14)
plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.savefig(
    "../images/roc_curve_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
auc_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "AUC Score": [
        auc_lr,
        auc_dt,
        auc_rf,
        auc_xgb
    ]
})
print(auc_results)

## 13. Final Model Evaluation & Threshold Optimization

XGBoost is the strongest performer, so it's selected as the final model. First, its performance at the default 0.5 threshold:

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

y_pred_final = best_xgb.predict(X_test_final)
y_prob_final = best_xgb.predict_proba(X_test_final)[:, 1]

print("Final XGBoost Accuracy:", accuracy_score(y_test, y_pred_final))
print("\n")
print("Final XGBoost Classification Report:")
print(classification_report(y_test, y_pred_final))

print("Final XGBoost ROC-AUC:", roc_auc_score(y_test, y_prob_final))

Since churn is imbalanced, the default 0.5 threshold isn't necessarily optimal. Using 5-fold
cross-validated out-of-fold predictions on the training set, different thresholds are swept to find
the one that maximizes F1 for the churn class.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
import numpy as np
import matplotlib.pyplot as plt

# Create 5-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Get out-of-fold probabilities for the churn class
y_prob_oof = cross_val_predict(
    best_xgb,
    X_train_final,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Test different probability thresholds
thresholds = np.arange(0.10, 0.91, 0.01)

f1_scores = []

for threshold in thresholds:
    y_pred_threshold = (y_prob_oof >= threshold).astype(int)
    f1 = f1_score(y_train, y_pred_threshold)
    f1_scores.append(f1)

# Find threshold with highest F1-score
best_index = np.argmax(f1_scores)
best_threshold = thresholds[best_index]
best_f1 = f1_scores[best_index]

print("Best Threshold:", round(best_threshold, 2))
print("Best Cross-Validated F1 Score:", round(best_f1, 4))

Applying the optimized threshold to the held-out test set:

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Apply optimized threshold to the test set
y_pred_optimized = (y_prob_final >= best_threshold).astype(int)

print("Results with optimized threshold:")
print(classification_report(y_test, y_pred_optimized))

print("Optimized Accuracy:",
      accuracy_score(y_test, y_pred_optimized))

print("Optimized Churn F1:",
      f1_score(y_test, y_pred_optimized))

## 14. Save Deployment Artifact

A final check of the exact feature set the model expects at inference time:

In [ ]:
print("Number of features:", len(X_train_final.columns))
print("\nFeatures expected by the model:")
print(list(X_train_final.columns))

Bundling the tuned XGBoost model together with its preprocessing objects (scaler, encoder) and the optimized threshold, so it can be loaded and used as a single artifact for deployment.

In [ ]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

feature_columns = list(X_train_final.columns)
deployment_artifact = {
    "model": best_xgb,
    "scaler": scaler,
    "ordinal_encoder": oe,
    "feature_columns": feature_columns,
    "continuous_cols": continuous_cols,
    "threshold": best_threshold
}
joblib.dump(
    deployment_artifact,
    "../models/churn_model.pkl"
)

print("Model and preprocessing saved successfully!")
print("Features:", feature_columns)
print("Threshold:", best_threshold)